## Downloading and preprocess the corpus

In [1]:
import nltk
nltk.download("brown")
nltk.download("webtext")
nltk.download("reuters")
nltk.download("punkt_tab")
from nltk.corpus import brown, webtext, reuters
brown_corpus = brown.sents()
brown_corpus = [" ".join(sentence) for sentence in brown_corpus]
brown_corpus = ["<s> " + sentence + " </s>" for sentence in brown_corpus][:5000]
webtext_corpus = webtext.sents()
webtext_corpus = [" ".join(sentence) for sentence in webtext_corpus]
webtext_corpus = ["<s> " + sentence + " </s>" for sentence in webtext_corpus][:5000]
reuters_corpus = reuters.sents()
reuters_corpus = [" ".join(sentence) for sentence in reuters_corpus]
reuters_corpus = ["<s> " + sentence + " </s>" for sentence in reuters_corpus][:5000]

[nltk_data] Downloading package brown to /Users/mukund/nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package webtext to /Users/mukund/nltk_data...
[nltk_data]   Package webtext is already up-to-date!
[nltk_data] Downloading package reuters to /Users/mukund/nltk_data...
[nltk_data]   Package reuters is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/mukund/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [3]:
from collections import Counter
import string
translator = str.maketrans('', '', string.punctuation)

## Q1

In [12]:
brown_corpus[0]

"<s> The Fulton County Grand Jury said Friday an investigation of Atlanta's recent primary election produced `` no evidence '' that any irregularities took place . </s>"

In [43]:
def calc_unigram(corpus):
    all_words=[]
    for sentence in corpus:
        sentence=sentence.replace('<s>', '').replace('</s>', '')
        sentence=sentence.lower()
        sentence=sentence.translate(translator)
        words=sentence.split()
        for word in words:
            all_words.append(word)

    word_counts = Counter(all_words)
    total_words=len(word_counts)

    probabilities={word : count/total_words for word, count in word_counts.items()}

    
    return probabilities

def get_unigram(word, corpus):
    probabilities=calc_unigram(corpus)
    return probabilities.get(word)

In [47]:
get_unigram("the", brown_corpus)

0.5184355874011046

## Q2

In [52]:
brown_corpus[0]

"<s> The Fulton County Grand Jury said Friday an investigation of Atlanta's recent primary election produced `` no evidence '' that any irregularities took place . </s>"

### Part (i)

In [5]:
all_words=[]

In [7]:
def get_bigrams(corpus):
    bigrams=[]
    for sentence in corpus:
        sentence=sentence.lower()
        words=sentence.split()
        for word in words:
            all_words.append(word)
    bigrams=list(zip(all_words[:-1], all_words[1:]))
    return bigrams

In [92]:
get_bigrams(["<s> my name is </s>"])

[('<s>', 'my'), ('my', 'name'), ('name', 'is'), ('is', '</s>')]

In [9]:
bigrams=get_bigrams(brown_corpus)

### Part ii

In [12]:
# Size of vocabulary = V
vocab=Counter(all_words)
V=len(vocab)
print(V)
N=len(all_words)
print(N)

13737
118731


In [53]:
def calc_bigram(word1, word2, bigrams):
    occurances=bigrams.count((word1, word2))
    N_w1=Counter(word1 for word1, _ in bigrams)
    w1=N_w1[word1]
    return (occurances+1)/(w1+V)

In [16]:
def calc_probabilities(bigrams):
    probabilities={}
    for word1 in vocab:
        for word2 in vocab:
            prob = calc_bigram(word1, word2, bigrams)
            probabilities[(word1, word2)] = prob
    return probabilities

In [55]:
def calc_prob1(bigrams, word1, word2):
    prob=calc_bigram(word1, word2, bigrams)
    return prob

In [171]:
calc_prob1(bigrams, "This", "is")

7.279609812914028e-05

### Part iii

In [154]:
def predict_next(word1, bigrams):
    b_counter=Counter(bigrams)
    N_w1=Counter(word1 for word1, _ in bigrams)
    w1=N_w1[word1]
    pred=None
    max_p=0
    for word2 in vocab:
        c_w1w2=b_counter[(word1, word2)]
        p=(c_w1w2+1)/(V+w1)
        if p > max_p:
            max_p=p
            pred=word2
    return pred

In [167]:
predict_next("this", bigrams)

'<s>'

### Part iv

In [189]:
def predict_sentence(initial, bigrams, limit=5):
    words=initial.lower().split()
    for i in range(limit-len(words)):
        next_word=predict_next(words[-1], bigrams)
        if not next_word:
            break
        words.append(next_word)
    return " ".join(words)

In [197]:
predict_sentence("and", bigrams, limit=10)

'and the first time . </s> <s> the first time'